# 01 Cleaning: from raw positions to one clean dataset

We combine the three raw position files, remove unusable rows, exclude positions
affected by suspicious clocks, remove rows without a valid label, and save
`cleaned_all.csv`.

Each row represents the position immediately before one played move.

## Cleaning rules

- We retain `game_id` for game-level splitting and cross-validation. It must not
  enter the model as a predictor.
- We retain `move_number` as a predictor of game stage. `ply` is used for cleaning
  checks but removed from the exported dataset.
- We remove diagnostic rows, rows with missing identifiers and duplicate
  `(game_id, ply)` observations.
- From the first suspicious clock observation onward, we exclude the remaining
  positions from that game. Earlier valid positions remain.
- Expected opening missingness is retained: pre-turn clocks are unknown on each
  player's first turn, and previous-move duration is unknown on their first two
  turns.
- We never fill missing `y`. The original raw data contains 80 unclassified
  `inconsistent_mate_evaluations` rows.
- Missing centipawn evaluations and mate values remain missing. Valid labeled
  rows are retained, and `pre_eval_cp_missing` records missing pre-move evaluation.
- Negative evaluation losses and extreme evaluations are retained and reported.
- No imputation or scaling happens here. Any learned preprocessing is fitted
  later inside the training folds.

We save the cleaned dataset separately and preserve the raw input files.

In [47]:
#import
import os

import numpy as np
import pandas as pd

## 1. Data import

We read the three raw files (1+0, 5+0 and 10+0 games) and stack them into one dataset `master`. The column `source_pgn` tells us which file a row comes from.

In [48]:
#one raw file per time control
files = ["games_1m.csv", "games_5m.csv", "games_10m.csv"]

#read the three files and stack them into master
parts = []
for file in files:
    parts.append(pd.read_csv("../data/600000_Databasis/" + file, dtype={"game_id": str}))
master = pd.concat(parts, ignore_index=True)

print(f"{master['game_id'].nunique()} games, {len(master)} positions, {master.shape[1]} columns")

9000 games, 586827 positions, 86 columns


In [49]:
#games and positions per file
master.groupby("source_pgn", sort=False).agg(games=("game_id", "nunique"), positions=("ply", "size"))

,games,positions
source_pgn,,
games_1m.pgn,3000,194221
games_5m.pgn,3000,202576
games_10m.pgn,3000,190030


## 2. Remove unusable rows

A row is unusable if the game could not be read (`pgn_error`), if it has no game id, ply or player, or if the same position (`game_id`, `ply`) appears twice. In our data there are none of these rows, but the checks stay so that the notebook also works on new data.

The file also has an empty separator column `" "` between the features and the metadata. We drop it.

In [50]:
#rows where the pgn could not be read
pgn_error = master["label_status"] == "pgn_error"
master = master.loc[~pgn_error]

#rows without game id, ply or a valid player
invalid = master["game_id"].isna() | master["ply"].isna() | ~master["player_to_move"].isin(["white", "black"])
master = master.loc[~invalid]

#the same position twice: keep the first one
duplicated = master.duplicated(subset=["game_id", "ply"], keep="first")
master = master.loc[~duplicated]

#empty separator column
master = master.drop(columns=" ")

#infinite values count as missing
master = master.replace([np.inf, -np.inf], np.nan)

print(f"removed {pgn_error.sum()} pgn errors, {invalid.sum()} invalid rows, {duplicated.sum()} duplicates")

removed 0 pgn errors, 0 invalid rows, 0 duplicates


In [51]:
#each game must be one block of rows in move order (the clock checks below rely on this)
assert (master["game_id"] != master["game_id"].shift()).sum() == master["game_id"].nunique()
assert (master.groupby("game_id")["ply"].diff() > 0).sum() == len(master) - master["game_id"].nunique()

## 3. Drop positions with suspicious clocks

All games use zero increment (1+0, 5+0 and 10+0). We treat the following clock values as suspicious:

- negative remaining time;
- remaining time above the nominal starting time, or an unknown starting time;
- an increase compared with the same player's previous known clock;
- negative previous-move duration;
- missing clocks or previous-move durations beyond the expected opening turns.

Expected opening missingness is retained: a player's pre-turn clock is unknown on their first turn, and their previous-move duration is unknown on their first two turns.

We check White and Black separately, using only information available up to the current position. Once either player's clock becomes suspicious, we exclude that position and all subsequent positions from that game. Earlier valid positions remain.

This is a conservative exclusion rule: clock adjustments can occur (e.g. on Lichess you can "gift" time to your opponnent), so a flagged value is not necessarily a recording error. We do not impute clocks during cleaning.

In [52]:
# Reconstruct each player's available pre-move clock.
white_to_move = master["player_to_move"] == "white"

clocks = {
    "white": master["time_left_seconds"].where(
        white_to_move, master["opponent_time_left_seconds"]
    ),
    "black": master["opponent_time_left_seconds"].where(
        white_to_move, master["time_left_seconds"]
    ),
}

games = master["game_id"]
start = master["initial_seconds"]
reasons = pd.DataFrame(index=master.index)

for color, clock in clocks.items():
    # Previous known clock: forward fill uses earlier values only.
    previous = clock.groupby(games).ffill().groupby(games).shift(1)

    # Before White's first move neither clock is known.
    # Before Black's first move only White's clock is known.
    opening_missing = master["ply"] <= (1 if color == "white" else 2)
    mover = master["player_to_move"] == color

    reasons[color + "_negative_clock"] = clock < 0
    reasons[color + "_above_start"] = clock > start
    reasons[color + "_no_start_time"] = clock.notna() & start.isna()
    reasons[color + "_clock_went_up"] = clock > previous
    reasons[color + "_unexpected_missing_clock"] = (
        clock.isna() & ~opening_missing
    )
    reasons[color + "_negative_move_time"] = (
        mover & (master["last_move_seconds"] < 0)
    )
    reasons[color + "_unexpected_missing_move_time"] = (
        mover
        & (master["ply"] > 4)
        & master["last_move_seconds"].isna()
    )

# Exclude from the first suspicious position onward within each game.
flagged = reasons.any(axis=1)
clock_dropped = flagged.groupby(games).cummax()

# Report exclusions before removing the rows.
print(f"{clock_dropped.sum():,} positions excluded because of clocks")
print(master.loc[clock_dropped].groupby("source_pgn").size())

master = master.loc[~clock_dropped].copy()

3,962 positions excluded because of clocks
source_pgn
games_10m.pgn    1294
games_1m.pgn      617
games_5m.pgn     2051
dtype: int64


## 4. Drop rows without a label

`y = 1` if the move loses more than 200 centipawns, plus the mate rules (missed forced win, allowed forced loss). For 80 positions Stockfish gave inconsistent mate evaluations, so there is no `y`. We drop these rows and never fill in `y`.

We drop them only now, because the clock checks in section 3 also need these moves to know the previous clock.

In [53]:
#which rows have no label
print(master.loc[master["y"].isna(), "label_status"].value_counts())

master = master.dropna(subset=["y"])
master["y"] = master["y"].astype(int)

label_status
inconsistent_mate_evaluations    80
Name: count, dtype: int64


## 5. Missing engine values

`pre_eval_cp` (evaluation before the move, in centipawns) is missing when Stockfish sees a forced mate: then there is a mate distance (`pre_mate`) instead of a centipawn value. The same holds for `played_eval_cp` and `played_mate`. These rows have a valid `y`, so we keep them.

We do not fill in `pre_eval_cp`. We only add the indicator `pre_eval_cp_missing`, so the model can see that this position was a mate position.

In [54]:
#1 = pre_eval_cp is missing
master["pre_eval_cp_missing"] = master["pre_eval_cp"].isna().astype(int)

master[["pre_eval_cp", "played_eval_cp", "pre_mate", "played_mate"]].isna().sum()

pre_eval_cp        25303
played_eval_cp     28409
pre_mate          557482
played_mate       554376
dtype: int64

## 6. Negative losses and large evaluations: only report
`eval_loss_cp` is the best-move evaluation minus the played-move evaluation, both from the mover’s perspective. A true loss relative to optimal play cannot be negative. However, our values come from two separate searches with limited node budgets. The restricted played-move search can therefore return a higher evaluation than the unrestricted search. A negative loss indicates disagreement between these estimates, rather than a demonstrated improvement over optimal play.  

There are 38,758 negative losses, with a mean of −25.58cp and a median of −11cp. Most disagreements are therefore small. Only 291 positions have a negative loss of −300cp or lower, representing approximately 0.05% of retained observations.  

We considered excluding these larger disagreements but retain them to avoid introducing an arbitrary cutoff that selectively removes non-blunder labels. Their retention does not establish that the evaluations are correct; uncertainty from limited-depth engine searches will always remain a issue. Large absolute evaluations can also occur in strongly unbalanced positions and are not, by themselves, evidence of corrupted data. We therefore decided to retain them.

In [55]:
#negative eval loss and very large evaluations (kept)
negative_loss = master["eval_loss_cp"] < 0
large_eval = master["pre_eval_cp"].abs() > 1000
print(f"{negative_loss.sum()} positions with negative eval loss, smallest {master['eval_loss_cp'].min():.0f} cp")
print(f"{large_eval.sum()} positions with |pre_eval_cp| > 1000 cp")

master["eval_loss_cp"].describe()
master.loc[negative_loss, "eval_loss_cp"].describe()

38758 positions with negative eval loss, smallest -1354 cp
18171 positions with |pre_eval_cp| > 1000 cp


count    38758.000000
mean       -25.579906
std         55.412467
min      -1354.000000
25%        -25.000000
50%        -11.000000
75%         -5.000000
max         -1.000000
Name: eval_loss_cp, dtype: float64

## 7. Save the clean dataset

Last checks, then we save one file. `pipeline_models.ipynb` reads exactly this file.

In [56]:
#sanity checks
assert not master.duplicated(subset=["game_id", "ply"]).any()
assert set(master["y"].unique()) == {0, 1}
assert (master["pre_eval_cp_missing"] == master["pre_eval_cp"].isna()).all()

os.makedirs("../data/processed", exist_ok=True)


#metadata has to be dropped  for the models (game_id and source_pgn are used for the split)
metadata = ["ply", "fen", "white_elo", "black_elo", "player_to_move", "utc_date", "utc_time",
            "played_move", "engine_best_move", "before_eval_depth", "played_eval_depth",
            "before_eval_nodes", "played_eval_nodes", "best_eval_status", "played_eval_status",
            "played_eval_cp", "pre_mate", "played_mate", "eval_loss_cp", "label_status", "pgn_error"]

master.drop(columns=metadata).to_csv("../data/processed/cleaned_all.csv", index=False)

print(f"{master['game_id'].nunique()} games, {len(master)} positions, blunder rate {master['y'].mean():.3f}")

9000 games, 582785 positions, blunder rate 0.115


## 8. Cleaning summary

In [57]:
#missing values per key column
key_columns = ["y", "pre_eval_cp", "played_eval_cp", "pre_mate", "played_mate",
               "time_left_seconds", "opponent_time_left_seconds", "last_move_seconds"]
missing = pd.DataFrame({"missing": master[key_columns].isna().sum(),
                        "share": master[key_columns].isna().mean().round(3)})
missing

,missing,share
y,0,0.000
pre_eval_cp,25303,0.043
played_eval_cp,28409,0.049
pre_mate,557482,0.957
played_mate,554376,0.951
time_left_seconds,17999,0.031
opponent_time_left_seconds,9000,0.015
last_move_seconds,35955,0.062


In [58]:
# Counts and blunder rates after all cleaning steps.
summary = master.groupby("source_pgn", sort=False).agg(
    games=("game_id", "nunique"),
    positions=("y", "size"),
    blunders=("y", "sum"),
    blunder_rate=("y", "mean"),
    missing_pre_eval=("pre_eval_cp_missing", "sum"),
    negative_loss=("eval_loss_cp", lambda s: (s < 0).sum()),
)

print(f"{master['game_id'].nunique():,} games retained")
print(f"{len(master):,} positions retained")

summary.round(3)

9,000 games retained
582,785 positions retained


,games,positions,blunders,blunder_rate,missing_pre_eval,negative_loss
source_pgn,,,,,,
games_1m.pgn,3000,193574,23029,0.119,8492,12983
games_5m.pgn,3000,200498,22229,0.111,8273,13359
games_10m.pgn,3000,188713,21638,0.115,8538,12416
